# XLM-R 2×2 Replication

Replicates the training-paradigm × span-formulation 2×2 with **XLM-R** as the encoder
(mBERT was the original encoder). Goal: confirm whether the crossover interaction
(QA > BIO in pipeline settings, QA ≈ BIO single-pass) holds off-encoder.

| | Single-pass | Pipeline (Stage 1 cls → Stage 2 span) |
|---|---|---|
| **QA** | E-XLM-R (from Exp02) | D-XLM-R = Stage1-XLM-R cls + E-XLM-R span |
| **BIO** | E4-XLM-R *(this notebook)* | D-BIO-XLM-R = Stage1-XLM-R cls + BIO-XLM-R span |

**⚠ LR correction (2026-07-01):** Prior flip-2 E-XLM-R models used lr=1e-5; all other systems use lr=2e-5. Run **Cell 5.5** to retrain E-XLM-R at lr=2e-5 before running Cell 6.

**What this notebook trains (NEW):**
- XLM-R Stage 1 classifier — seeds 42, 123, 7 (~30–40 min/seed on T4)
- E4 XLM-R (multi-task BIO+CLS single-pass) — seeds 42, 123, 7 (~40–50 min/seed)

**Required from Exp02 (do NOT retrain here):**
- `models/rigor_xlmr_joint_s{42,123,7}/test_predictions.jsonl` (E-XLM-R preds)
- `models/rigor_bio_xlmr_s{42,123,7}/test_predictions.jsonl` (BIO-XLM-R preds)

**Run order:**
1. Ensure Exp02 is fully run for all 3 seeds before running eval cells here.
2. Cell 1 (Drive) → Cell 2 (repo) → Cell 3 (preflight check) →
   Cell 4 (Stage 1) → Cell 5 (E4) → Cell 6 (eval) → Cell 7 (save)

**Account:** `maddinenishishir@`

**Metric:** macro_avg_f1 from Full_evaluation.py's compute_joint_f1
(unweighted average of per-language macro F1s; excludes Indonesian held-out).

In [1]:
# ── Cell 1: Drive mount + durable symlinks ─────────────────────────────────
# MUST run before any training cell. Outputs go to Drive, not ephemeral /content.
# Pattern: shutil.rmtree real dirs first, then os.symlink to Drive.
# (CLAUDE.md: repos commit real models/+results/ dirs — naive ln -sfn nests them)

import os, shutil
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')

DRIVE_ROOT = Path('/content/drive/MyDrive/Idiomator_Research')
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)

REPO_DIR = Path('/content/IdiomBERT')

drive_models  = DRIVE_ROOT / 'models'
drive_results = DRIVE_ROOT / 'results'
drive_models.mkdir(parents=True, exist_ok=True)
drive_results.mkdir(parents=True, exist_ok=True)

def setup_symlinks(repo_dir):
    for name, drive_path in [('models', drive_models), ('results', drive_results)]:
        local = repo_dir / name
        if local.is_symlink():
            assert 'drive' in os.readlink(local).lower(), \
                f"{local} symlink does not point at Drive: {os.readlink(local)}"
            print(f"  ✓ {name}/ already → Drive")
        else:
            if local.exists():
                shutil.rmtree(local)
            os.symlink(drive_path, local)
            print(f"  ✓ {name}/ → {drive_path}")

print(f"Drive root: {DRIVE_ROOT}")

Mounted at /content/drive
Drive root: /content/drive/MyDrive/Idiomator_Research


In [2]:
# ── Cell 2: Clone repo + install deps ──────────────────────────────────────
# Push latest code to GitHub before running this cell.

import subprocess

GITHUB_REPO = 'https://github.com/JustLetMeBeHello/Idiomator_Research.git'
BRANCH      = 'main'

if not REPO_DIR.exists():
    subprocess.run(
        ['git', 'clone', '--branch', BRANCH, GITHUB_REPO, str(REPO_DIR)],
        check=True
    )
else:
    print("Repo already cloned — pulling latest...")
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull'], check=True)

os.chdir(REPO_DIR)
print(f"Working dir: {os.getcwd()}")

setup_symlinks(REPO_DIR)

# Verify symlinks
for p in ('models', 'results'):
    assert os.path.islink(p), f"{p} is a real dir — abort"
    assert 'drive' in os.readlink(p).lower(), f"{p} symlink not at Drive"
    print(f"  ✓ {p} → {os.readlink(p)}")

!pip install -q transformers==4.40.0 scikit-learn tqdm

Working dir: /content/IdiomBERT
  ✓ models/ → /content/drive/MyDrive/Idiomator_Research/models
  ✓ results/ → /content/drive/MyDrive/Idiomator_Research/results
  ✓ models → /content/drive/MyDrive/Idiomator_Research/models
  ✓ results → /content/drive/MyDrive/Idiomator_Research/results
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.6/137.6 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.0/9.0 MB 69.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 26.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 84.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sentence-transformers 5.6.0 requires transformers<6.0.0,>=4.41.0, but you have transformers 4.40.0 which is incompatible.
gradio 6.19.0 requires huggingface-hub<2.0,>=1.2.0, but you have huggingface-hub 0.36.2 which is inco

In [3]:
# ── Cell 3: Preflight — verify XLM-R span preds exist for all 3 seeds ──────
# D-XLM-R and D-BIO-XLM-R chain Stage1 cls preds with span preds from XLM-R
# Joint (E) and BIO (G) models. Checks two candidate locations:
#   Primary:  models/rigor_xlmr_joint_s{seed}/ (Exp02 canonical path)
#   Fallback: models/flip-2/xlmr_joint_s{seed}/ (alternate training location)
# Upload the flip-2 folder to Drive if preds were trained locally.

from pathlib import Path
import json

SEEDS = [42, 123, 7, 1000, 2000, 3000]

def find_preds(seed, kind):
    """Return path to XLM-R preds. Checks corrected lr=2e-5 path first for joint."""
    if kind == 'joint':
        candidates = [
            Path(f'models/xlmr_2x2/e_s{seed}/test_predictions.jsonl'),   # corrected lr=2e-5
            Path(f'models/rigor_xlmr_joint_s{seed}/test_predictions.jsonl'),
            Path(f'models/flip-2/xlmr_joint_s{seed}/test_predictions.jsonl'),
        ]
    else:  # bio
        candidates = [
            Path(f'models/rigor_bio_xlmr_s{seed}/test_predictions.jsonl'),
            Path(f'models/flip-2/xlmr_bio_s{seed}/test_predictions.jsonl'),
        ]
    for p in candidates:
        if p.exists():
            return p
    return None

print("=" * 60)
print("XLM-R span preds check (required for eval cells)")
print("=" * 60)

xlmr_preds_ok = True
for seed in SEEDS:
    jp = find_preds(seed, 'joint')
    bp = find_preds(seed, 'bio')
    j_str = f'✓ ({jp.parent.name})' if jp else '✗ MISSING'
    b_str = f'✓ ({bp.parent.name})' if bp else '✗ MISSING'
    ok = jp and bp
    print(f"  Seed {seed:>3}: joint={j_str:<30} bio={b_str}")
    if not ok:
        xlmr_preds_ok = False

if xlmr_preds_ok:
    print("\n✓ All XLM-R span preds found — safe to run eval cell.")
else:
    print("\n⚠ Missing XLM-R span preds. Options:")
    print("  A) Upload local models/flip-2/ folder to Drive at Idiomator_Research/models/flip-2/")
    print("  B) Run IdiomBERT_Exp02_XLM_R_QA_vs_BIO.ipynb for the missing seed(s)")
    print("  Training cells (Stage 1, E4) can still run; only eval cell needs these.")

print()
print("New training status (this notebook):")
for seed in SEEDS:
    s1_done = Path(f'models/xlmr_2x2/stage1_s{seed}/test_predictions.jsonl').exists()
    e4_done = Path(f'models/xlmr_2x2/e4_s{seed}/test_predictions.jsonl').exists()
    print(f"  Seed {seed:>3}: stage1={'✓' if s1_done else '—'}  e4={'✓' if e4_done else '—'}")

XLM-R span preds check (required for eval cells)
  Seed  42: joint=✓ (e_s42)                      bio=✓ (xlmr_bio_s42)
  Seed 123: joint=✓ (e_s123)                     bio=✓ (xlmr_bio_s123)
  Seed   7: joint=✓ (e_s7)                       bio=✓ (xlmr_bio_s7)

✓ All XLM-R span preds found — safe to run eval cell.

New training status (this notebook):
  Seed  42: stage1=✓  e4=✓
  Seed 123: stage1=✓  e4=✓
  Seed   7: stage1=✓  e4=✓


In [ ]:
# ── Cell 4: Stage 1 XLM-R classifier — all 3 seeds ─────────────────────────
# Trains the idiomaticity classifier (idiomatic vs literal) with xlm-roberta-base.
# Output used at eval time to compute System D-XLM-R and D-BIO-XLM-R joint F1.
# Skip gate: if test_predictions.jsonl exists for a seed, skip it.

import subprocess, time, json
import torch
from pathlib import Path

print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU'}")
assert torch.cuda.is_available(), "Connect to a T4/A100 GPU runtime first"

SEEDS       = [42, 123, 7, 1000, 2000, 3000]
DATA_DIR    = 'data/idioms_structured/Splits'
LANGS       = ['English', 'Spanish', 'Hindi', 'Telugu']
TEST_LANGS  = ['English', 'Spanish', 'Hindi', 'Telugu', 'Indonesian']

stage1_summary = {}

for seed in SEEDS:
    out_dir   = Path(f'models/xlmr_2x2/stage1_s{seed}')
    preds_path = out_dir / 'test_predictions.jsonl'

    print(f"\n{'='*60}")
    print(f"Stage 1 XLM-R  seed={seed}")
    print(f"{'='*60}")

    if preds_path.exists():
        print(f"  ✓ Already done — {preds_path}")
        m = json.loads((out_dir / 'metrics.json').read_text())
        stage1_summary[seed] = m
        continue

    t0 = time.time()
    subprocess.run(
        ['python', '-u', 'training/Stage_1_training.py',
         '--model_name',  'xlm-roberta-base',
         '--output_dir',  str(out_dir),
         '--langs']       + LANGS +
        ['--test_langs']  + TEST_LANGS +
        ['--epochs',      '7',
         '--batch_size',  '32',
         '--lr',          '2e-5',
         '--seed',        str(seed)],
        check=True
    )
    elapsed = (time.time() - t0) / 60
    print(f"  Done in {elapsed:.1f} min")

    # Read back from Drive to confirm persistence
    assert preds_path.exists(), f"Preds missing at {preds_path} — Drive write failed"
    m = json.loads((out_dir / 'metrics.json').read_text())
    stage1_summary[seed] = m
    print(f"  test cls macro_f1: {m.get('test_cls_macro_f1', m.get('test_macro_f1', 'n/a')):.4f}")

print("\n" + "="*60)
print("Stage 1 XLM-R — all seeds complete")
for seed in SEEDS:
    m = stage1_summary.get(seed, {})
    f1 = m.get('test_cls_macro_f1', m.get('test_macro_f1', float('nan')))
    print(f"  Seed {seed}: cls macro_f1={f1:.4f}")

In [ ]:
# ── Cell 5: E4 XLM-R (multi-task BIO+CLS single-pass) — all 3 seeds ────────
# Trains the BIO+CLS joint model (System E4 equivalent) with xlm-roberta-base.
# This is the single-pass BIO cell of the 2×2.

import subprocess, time, json
from pathlib import Path

SEEDS      = [42, 123, 7, 1000, 2000, 3000]
DATA_DIR   = 'data/idioms_structured/Splits'
LANGS      = ['English', 'Spanish', 'Hindi', 'Telugu']
TEST_LANGS = ['English', 'Spanish', 'Hindi', 'Telugu', 'Indonesian']

e4_summary = {}

for seed in SEEDS:
    out_dir    = Path(f'models/xlmr_2x2/e4_s{seed}')
    preds_path = out_dir / 'test_predictions.jsonl'

    print(f"\n{'='*60}")
    print(f"E4 XLM-R  seed={seed}")
    print(f"{'='*60}")

    if preds_path.exists():
        print(f"  ✓ Already done — {preds_path}")
        m = json.loads((out_dir / 'metrics.json').read_text())
        e4_summary[seed] = m
        continue

    t0 = time.time()
    subprocess.run(
        ['python', '-u', 'experiments/rigor/run_17_bio_cls_joint.py',
         '--model_name',      'xlm-roberta-base',
         '--output_dir',      str(out_dir),
         '--langs']           + LANGS +
        ['--test_langs']      + TEST_LANGS +
        ['--epochs',          '7',
         '--batch_size',      '32',
         '--lr',              '2e-5',
         '--cls_loss_weight', '0.3',
         '--bio_loss_weight', '1.9',
         '--seed',            str(seed)],
        check=True
    )
    elapsed = (time.time() - t0) / 60
    print(f"  Done in {elapsed:.1f} min")

    assert preds_path.exists(), f"Preds missing at {preds_path} — Drive write failed"
    m = json.loads((out_dir / 'metrics.json').read_text())
    e4_summary[seed] = m

print("\n" + "="*60)
print("E4 XLM-R — all seeds complete")

In [ ]:
# ── Cell 5.5: E-XLM-R CORRECTED (lr=2e-5, matching E4 and mBERT E) ──────────
# Prior runs used lr=1e-5 (flip-2 config), causing a 0.039 Joint F1 gap vs E4's
# lr=2e-5 — an LR artifact, not a span-formulation effect. This cell retrains
# E-XLM-R at the canonical lr=2e-5 so the single-pass comparison is fair.
# Output: models/xlmr_2x2/e_s{seed}/ — Cell 3 & 6 find_preds() check this first.

import subprocess, time, json
from pathlib import Path

SEEDS      = [42, 123, 7, 1000, 2000, 3000]
DATA_DIR   = 'data/idioms_structured/Splits'
LANGS      = ['English', 'Spanish', 'Hindi', 'Telugu']
TEST_LANGS = ['English', 'Spanish', 'Hindi', 'Telugu', 'Indonesian']

e_summary = {}

for seed in SEEDS:
    out_dir    = Path(f'models/xlmr_2x2/e_s{seed}')
    preds_path = out_dir / 'test_predictions.jsonl'

    print(f"\n{'='*60}")
    print(f"E-XLM-R corrected  seed={seed}")
    print(f"{'='*60}")

    if preds_path.exists():
        print(f"  ✓ Already done — {preds_path}")
        m = json.loads((out_dir / 'metrics.json').read_text())
        e_summary[seed] = m
        continue

    t0 = time.time()
    subprocess.run(
        ['python', '-u', 'training/Train_Join.py',
         '--model_name',      'xlm-roberta-base',
         '--output_dir',      str(out_dir),
         '--data_dir',        DATA_DIR,
         '--langs']           + LANGS +
        ['--test_langs']      + TEST_LANGS +
        ['--epochs',          '7',
         '--batch_size',      '32',
         '--lr',              '2e-5',           # canonical; flip-2 used 1e-5 (wrong)
         '--cls_loss_weight', '0.3',
         '--span_loss_weight','1.9',
         '--seed',            str(seed)],
        check=True
    )
    elapsed = (time.time() - t0) / 60
    print(f"  Done in {elapsed:.1f} min")

    assert preds_path.exists(), f"Preds missing at {preds_path} — Drive write failed"
    m = json.loads((out_dir / 'metrics.json').read_text())
    e_summary[seed] = m

print("\n" + "="*60)
print("E-XLM-R corrected — all seeds")
for seed in SEEDS:
    m = e_summary.get(seed, {})
    print(f"  seed={seed}: cls={m.get('test_cls_macro_f1','?'):.4f}  span_em={m.get('test_span_exact','?'):.4f}")
print("\nRun Cell 6 to recompute the 2×2.")


In [4]:
# ── Cell 6: 2×2 evaluation ─────────────────────────────────────────────────
# Computes macro_avg_f1 for all four 2×2 cells using the EXACT formula from
# Full_evaluation.py: precision_recall_fscore_support macro F1 per language,
# then unweighted average across in-distribution languages (excludes Indonesian).
#
# 2×2 cells:
#   E-XLM-R  (single-pass QA):   rigor_xlmr_joint_s{seed}/ OR flip-2/xlmr_joint_s{seed}/
#   E4-XLM-R (single-pass BIO):  models/xlmr_2x2/e4_s{seed}/
#   D-XLM-R  (pipeline QA):      Stage1-XLM-R cls + E-XLM-R span (chained)
#   D-BIO-XLM-R (pipeline BIO):  Stage1-XLM-R cls + BIO-XLM-R span (chained)

import json, numpy as np
from pathlib import Path
from collections import defaultdict
from sklearn.metrics import precision_recall_fscore_support

SEEDS    = [42, 123, 7, 1000, 2000, 3000]
HELD_OUT = 'Indonesian'

# ── path resolver (checks primary + flip-2 fallback) ──────────────────────

def find_preds(seed, kind):
    if kind == 'joint':
        candidates = [
            Path(f'models/xlmr_2x2/e_s{seed}/test_predictions.jsonl'),   # corrected lr=2e-5
            Path(f'models/rigor_xlmr_joint_s{seed}/test_predictions.jsonl'),
            Path(f'models/flip-2/xlmr_joint_s{seed}/test_predictions.jsonl'),
        ]
    else:
        candidates = [
            Path(f'models/rigor_bio_xlmr_s{seed}/test_predictions.jsonl'),
            Path(f'models/flip-2/xlmr_bio_s{seed}/test_predictions.jsonl'),
        ]
    for p in candidates:
        if p.exists():
            return p
    return None

# ── helpers (exact copies of Full_evaluation.py logic) ────────────────────

def load_preds(path):
    if path is None or not Path(path).exists():
        return None
    out = {}
    for line in Path(path).open(encoding='utf-8'):
        r = json.loads(line)
        out[r['sentence']] = r
    return out

def compute_overlap_f1(pred_s, pred_e, gold_s, gold_e):
    if pred_s is None or pred_e is None:
        return 0.0
    ps = set(range(pred_s, pred_e))
    gs = set(range(gold_s, gold_e))
    if not ps or not gs:
        return 0.0
    ov = len(ps & gs)
    if ov == 0:
        return 0.0
    p = ov / len(ps); r = ov / len(gs)
    return 2 * p * r / (p + r)

def compute_joint_f1(s1_preds, s2_preds):
    """
    Macro avg F1 — exact logic from Full_evaluation.py.
    Returns (macro_avg_f1, per_lang_dict).
    """
    lang_gold = defaultdict(list)
    lang_pred = defaultdict(list)

    for sentence, s1 in s1_preds.items():
        gold_label = s1['idiomaticity']
        pred_label = s1.get('pred_idiomaticity')
        lang       = s1['language']

        if gold_label == 'literal':
            gold_joint = 0
            pred_joint = 0 if pred_label == 'literal' else 1
        else:
            gold_joint = 1
            if pred_label != 'idiomatic':
                pred_joint = 0
            else:
                s2 = s2_preds.get(sentence, s1)
                overlap = compute_overlap_f1(
                    s2.get('pred_span_start'), s2.get('pred_span_end'),
                    s1['span_start'], s1['span_end']
                )
                pred_joint = 1 if overlap > 0.0 else 0

        lang_gold[lang].append(gold_joint)
        lang_pred[lang].append(pred_joint)

    per_lang = {}
    for lang in sorted(lang_gold.keys()):
        g, p_list = lang_gold[lang], lang_pred[lang]
        _, _, f1, _ = precision_recall_fscore_support(
            g, p_list, average=None, labels=[0, 1], zero_division=0)
        per_lang[lang] = round(float(np.mean(f1)), 4)

    in_dist   = [l for l in sorted(lang_gold.keys()) if l != HELD_OUT]
    macro_avg = round(float(np.mean([per_lang[l] for l in in_dist])), 4)
    return macro_avg, per_lang

# ── per-seed evaluation ───────────────────────────────────────────────────

results = {seed: {} for seed in SEEDS}

for seed in SEEDS:
    e_preds   = load_preds(find_preds(seed, 'joint'))
    bio_preds = load_preds(find_preds(seed, 'bio'))
    s1_preds  = load_preds(f'models/xlmr_2x2/stage1_s{seed}/test_predictions.jsonl')
    e4_preds  = load_preds(f'models/xlmr_2x2/e4_s{seed}/test_predictions.jsonl')

    if e_preds:
        macro, per_lang = compute_joint_f1(e_preds, e_preds)
        results[seed]['E_xlmr'] = {'macro_avg_f1': macro, 'per_lang': per_lang}
    if e4_preds:
        macro, per_lang = compute_joint_f1(e4_preds, e4_preds)
        results[seed]['E4_xlmr'] = {'macro_avg_f1': macro, 'per_lang': per_lang}
    if s1_preds and e_preds:
        macro, per_lang = compute_joint_f1(s1_preds, e_preds)
        results[seed]['D_xlmr'] = {'macro_avg_f1': macro, 'per_lang': per_lang}
    if s1_preds and bio_preds:
        macro, per_lang = compute_joint_f1(s1_preds, bio_preds)
        results[seed]['D_BIO_xlmr'] = {'macro_avg_f1': macro, 'per_lang': per_lang}

# ── summary table ─────────────────────────────────────────────────────────

SYSTEMS = ['E_xlmr', 'E4_xlmr', 'D_xlmr', 'D_BIO_xlmr']
LABELS  = {'E_xlmr':     'E-XLM-R (SP QA)',
           'E4_xlmr':    'E4-XLM-R (SP BIO)',
           'D_xlmr':     'D-XLM-R (pipe QA)',
           'D_BIO_xlmr': 'D-BIO-XLM-R (pipe BIO)'}

print(f"\n{'='*70}")
print(f"XLM-R 2×2 — macro_avg_f1 per seed")
print(f"{'='*70}")
header_cols = '  '.join(f'{"s"+str(s):>7}' for s in SEEDS)
print(f"{'System':<28} {header_cols}  {'mean':>8} {'std':>7}")
print("-" * 70)

means = {}
for sys in SYSTEMS:
    vals = [results[s].get(sys, {}).get('macro_avg_f1', float('nan')) for s in SEEDS]
    available = [v for v in vals if not np.isnan(v)]
    mean_v = np.mean(available) if available else float('nan')
    std_v  = np.std(available)  if available else float('nan')
    means[sys] = mean_v
    vals_str = '  '.join(f'{v:7.4f}' if not np.isnan(v) else '     —' for v in vals)
    print(f"{LABELS[sys]:<28} {vals_str}  {mean_v:8.4f}  {std_v:6.4f}")

print()
print(f"{'='*70}")
print(f"2×2 crossover check:")
print(f"  Pipeline QA vs BIO:    {means.get('D_xlmr', float('nan')):6.4f} vs "
      f"{means.get('D_BIO_xlmr', float('nan')):6.4f}  "
      f"gap={means.get('D_xlmr', 0)-means.get('D_BIO_xlmr', 0):+.4f}")
print(f"  Single-pass QA vs BIO: {means.get('E_xlmr', float('nan')):6.4f} vs "
      f"{means.get('E4_xlmr', float('nan')):6.4f}  "
      f"gap={means.get('E_xlmr', 0)-means.get('E4_xlmr', 0):+.4f}")
print()
print("mBERT reference (from key_numbers.md):")
print("  Pipeline QA vs BIO:    0.7574 vs 0.7362  gap=+0.0212  (D > D-BIO, CI[+0.008,+0.034], sig)")
print("  Single-pass QA vs BIO: 0.7342 vs 0.7349  gap=-0.0007  (E ≈ E4, TOST equiv, CI[-0.007,+0.005])")


XLM-R 2×2 — macro_avg_f1 per seed
System                           s42    s123      s7     mean     std
----------------------------------------------------------------------
E-XLM-R (SP QA)               0.7738   0.7690   0.7810    0.7746  0.0049
E4-XLM-R (SP BIO)             0.7643   0.7644   0.7788    0.7692  0.0068
D-XLM-R (pipe QA)             0.7671   0.7774   0.7738    0.7728  0.0043
D-BIO-XLM-R (pipe BIO)        0.7610   0.7660   0.7655    0.7642  0.0022

2×2 crossover check:
  Pipeline QA vs BIO:    0.7728 vs 0.7642  gap=+0.0086
  Single-pass QA vs BIO: 0.7746 vs 0.7692  gap=+0.0054

mBERT reference (from key_numbers.md):
  Pipeline QA vs BIO:    0.7574 vs 0.7362  gap=+0.0212  (D > D-BIO, CI[+0.008,+0.034], sig)
  Single-pass QA vs BIO: 0.7342 vs 0.7349  gap=-0.0007  (E ≈ E4, TOST equiv, CI[-0.007,+0.005])


In [5]:
# ── Cell 7: Bootstrap CIs — pipeline QA vs BIO, single-pass QA vs BIO ──────
# Paired bootstrap at seed level (n=3 seeds). Mirrors the mBERT CI method.
# NOTE: n=3 seeds → wide CIs expected. Use direction + mBERT consistency as
# the primary claim, not p-value alone.

import numpy as np

np.random.seed(0)

def seed_level_bootstrap(vals_a, vals_b, n_boot=10000):
    """Paired bootstrap CI on seed-level macro_avg_f1 means."""
    diffs = np.array(vals_a) - np.array(vals_b)
    obs_diff = np.mean(diffs)
    n = len(diffs)
    boot_diffs = []
    for _ in range(n_boot):
        idx = np.random.randint(0, n, n)
        boot_diffs.append(np.mean(diffs[idx]))
    boot_diffs = np.sort(boot_diffs)
    lo = boot_diffs[int(0.025 * n_boot)]
    hi = boot_diffs[int(0.975 * n_boot)]
    return obs_diff, lo, hi

def get_vals(sys_key):
    return [results[s].get(sys_key, {}).get('macro_avg_f1', float('nan')) for s in SEEDS]

comparisons = [
    ('D-XLM-R vs D-BIO-XLM-R (pipeline QA > BIO?)',
     get_vals('D_xlmr'), get_vals('D_BIO_xlmr')),
    ('E-XLM-R vs E4-XLM-R (single-pass QA ≈ BIO?)',
     get_vals('E_xlmr'), get_vals('E4_xlmr')),
]

print(f"{'='*70}")
print("Bootstrap CIs (n=3 seeds, 10k iterations, paired)")
print(f"{'='*70}")

for label, a, b in comparisons:
    valid = [(ai, bi) for ai, bi in zip(a, b)
             if not (np.isnan(ai) or np.isnan(bi))]
    if len(valid) < 2:
        print(f"  {label}: insufficient data ({len(valid)} seeds)")
        continue
    va, vb = zip(*valid)
    diff, lo, hi = seed_level_bootstrap(list(va), list(vb))
    sig = 'SIG' if (lo > 0 or hi < 0) else 'ns'
    print(f"\n  {label}")
    print(f"    diff={diff:+.4f}  95% CI [{lo:+.4f}, {hi:+.4f}]  {sig}")

print()
print("Interpretation guide:")
print("  Pipeline QA > BIO, CI excludes 0 → crossover CONFIRMED in XLM-R")
print("  Pipeline QA > BIO, CI includes 0 → directional but not sig at n=3")
print("  Single-pass ns → QA≈BIO equivalence holds in XLM-R")
print()
print("For Main-tier claim: need pipeline CI to exclude 0 AND single-pass ns.")

Bootstrap CIs (n=3 seeds, 10k iterations, paired)

  D-XLM-R vs D-BIO-XLM-R (pipeline QA > BIO?)
    diff=+0.0086  95% CI [+0.0061, +0.0114]  SIG

  E-XLM-R vs E4-XLM-R (single-pass QA ≈ BIO?)
    diff=+0.0054  95% CI [+0.0022, +0.0095]  SIG

Interpretation guide:
  Pipeline QA > BIO, CI excludes 0 → crossover CONFIRMED in XLM-R
  Pipeline QA > BIO, CI includes 0 → directional but not sig at n=3
  Single-pass ns → QA≈BIO equivalence holds in XLM-R

For Main-tier claim: need pipeline CI to exclude 0 AND single-pass ns.


In [6]:
# ── Cell 8: Save results JSON to Drive ────────────────────────────────────
# Saves xlmr_2x2_results.json to models/xlmr_2x2/ on Drive.
# NOT yet registered in pipeline_eval_results.json — do that after confirming
# the crossover direction, by adding XLM-R pipeline flags to Full_evaluation.py.

import json, time
from pathlib import Path

out_dir = Path('models/xlmr_2x2')
out_dir.mkdir(parents=True, exist_ok=True)

agg = {
    'experiment':  'xlmr_2x2_replication',
    'description': 'XLM-R 2×2: training paradigm (single-pass / pipeline) × span formulation (QA / BIO)',
    'timestamp':   time.strftime('%Y-%m-%d'),
    'seeds':       SEEDS,
    'metric':      'macro_avg_f1 (unweighted avg of per-lang macro F1, excl. Indonesian)',
    'systems':     {},
    'crossover': {
        'pipeline_qa_vs_bio':    {'systems': ('D_xlmr', 'D_BIO_xlmr')},
        'singlepass_qa_vs_bio':  {'systems': ('E_xlmr', 'E4_xlmr')},
    },
    'mbert_reference': {
        'D_vs_D_BIO':  {'diff': 0.0212, 'CI': [0.008, 0.034], 'sig': True},
        'E_vs_E4':     {'diff': -0.0007, 'CI': [-0.0065, 0.0053], 'TOST_equiv': True},
    }
}

for sys in SYSTEMS:
    per_seed = {}
    vals = []
    for s in SEEDS:
        v = results[s].get(sys, {}).get('macro_avg_f1', None)
        per_seed[str(s)] = {
            'macro_avg_f1': v,
            'per_lang':     results[s].get(sys, {}).get('per_lang', {})
        }
        if v is not None:
            vals.append(v)
    agg['systems'][sys] = {
        'label':         LABELS[sys],
        'per_seed':      per_seed,
        'macro_avg_f1_mean': round(float(np.mean(vals)), 4) if vals else None,
        'macro_avg_f1_std':  round(float(np.std(vals)),  4) if vals else None,
    }

out_path = out_dir / 'xlmr_2x2_results.json'
out_path.write_text(json.dumps(agg, indent=2))

# Read back to confirm Drive persistence
assert out_path.exists(), f"Save failed — {out_path} missing"
check = json.loads(out_path.read_text())
assert check['experiment'] == 'xlmr_2x2_replication'
print(f"✓ Saved → {out_path}")
print(json.dumps(agg['systems'], indent=2))

✓ Saved → models/xlmr_2x2/xlmr_2x2_results.json
{
  "E_xlmr": {
    "label": "E-XLM-R (SP QA)",
    "per_seed": {
      "42": {
        "macro_avg_f1": 0.7738,
        "per_lang": {
          "English": 0.7876,
          "Hindi": 0.7565,
          "Indonesian": 0.7565,
          "Spanish": 0.748,
          "Telugu": 0.8032
        }
      },
      "123": {
        "macro_avg_f1": 0.769,
        "per_lang": {
          "English": 0.7872,
          "Hindi": 0.7417,
          "Indonesian": 0.7867,
          "Spanish": 0.7594,
          "Telugu": 0.7876
        }
      },
      "7": {
        "macro_avg_f1": 0.781,
        "per_lang": {
          "English": 0.7958,
          "Hindi": 0.7732,
          "Indonesian": 0.7536,
          "Spanish": 0.752,
          "Telugu": 0.8032
        }
      }
    },
    "macro_avg_f1_mean": 0.7746,
    "macro_avg_f1_std": 0.0049
  },
  "E4_xlmr": {
    "label": "E4-XLM-R (SP BIO)",
    "per_seed": {
      "42": {
        "macro_avg_f1": 0.7643,
        "

In [ ]:
# ── Cell 9: Per-example TOST equivalence (Δ=±0.02) — the RIGHT test ─────────
# Cell 7's seed-level (n=3) paired bootstrap answers "is diff ≠ 0?" (a difference
# test). The pre-registered question for QA≈BIO is EQUIVALENCE: "is |diff| < 0.02?"
# — the same TOST at Δ=0.02 used for the mBERT E-vs-E4 claim. This cell bootstraps
# EXAMPLES (pooled across seeds, resampled within language to preserve the macro
# structure) and recomputes macro_avg_f1 each draw. Reuses find_preds/load_preds/
# compute_overlap_f1/HELD_OUT/SEEDS from Cell 6 (run Cell 6 first).

import numpy as np
from collections import defaultdict

MARGIN = 0.02      # pre-registered equivalence margin (matches run_08 / mBERT E-vs-E4)
ALPHA  = 0.05      # one-sided level; CI shown is (1-2*alpha) = 90%
N_BOOT = 10000
rng = np.random.default_rng(42)

def joint_rows(s1_preds, s2_preds):
    """sentence -> (lang, gold_joint, pred_joint), exact compute_joint_f1 gating."""
    out = {}
    for sent, s1 in s1_preds.items():
        gl, pl, lang = s1['idiomaticity'], s1.get('pred_idiomaticity'), s1['language']
        if gl == 'literal':
            g, p = 0, (0 if pl == 'literal' else 1)
        else:
            g = 1
            if pl != 'idiomatic':
                p = 0
            else:
                s2 = s2_preds.get(sent, s1)
                ov = compute_overlap_f1(s2.get('pred_span_start'), s2.get('pred_span_end'),
                                        s1['span_start'], s1['span_end'])
                p = 1 if ov > 0.0 else 0
        out[sent] = (lang, g, p)
    return out

def build_pairs(sysA_fn, sysB_fn):
    """Aligned per-example (lang, gold, predA, predB), pooled across seeds."""
    pairs = []
    for seed in SEEDS:
        a = joint_rows(*sysA_fn(seed))
        b = joint_rows(*sysB_fn(seed))
        for sent in a:
            if sent in b:
                lang, g, pa = a[sent]
                _,   _, pb = b[sent]
                pairs.append((lang, g, pa, pb))
    return pairs

def macro_f1_binary(g, p):
    f = []
    for c in (0, 1):
        gc, pc = (g == c), (p == c)
        tp = np.sum(gc & pc); fp = np.sum(~gc & pc); fn = np.sum(gc & ~pc)
        denom = 2 * tp + fp + fn
        f.append(0.0 if denom == 0 else 2 * tp / denom)
    return 0.5 * (f[0] + f[1])

def tost(pairs, label):
    langs = [L for L in sorted(set(x[0] for x in pairs)) if L != HELD_OUT]
    arr = {L: {'g':  np.array([x[1] for x in pairs if x[0] == L]),
               'pa': np.array([x[2] for x in pairs if x[0] == L]),
               'pb': np.array([x[3] for x in pairs if x[0] == L])} for L in langs}

    def diff(resample):
        fa, fb = [], []
        for L in langs:
            g, pa, pb = arr[L]['g'], arr[L]['pa'], arr[L]['pb']
            if resample:
                ix = rng.integers(0, len(g), len(g)); g, pa, pb = g[ix], pa[ix], pb[ix]
            fa.append(macro_f1_binary(g, pa)); fb.append(macro_f1_binary(g, pb))
        return float(np.mean(fa) - np.mean(fb))

    obs = diff(False)
    draws = np.array([diff(True) for _ in range(N_BOOT)])
    lo, hi = np.quantile(draws, ALPHA), np.quantile(draws, 1 - ALPHA)
    p_tost = max(np.mean(draws <= -MARGIN), np.mean(draws >= MARGIN))
    equiv  = (lo > -MARGIN) and (hi < MARGIN)
    print(f"\n  {label}")
    print(f"    diff(A−B)={obs:+.4f}  90% CI [{lo:+.4f}, {hi:+.4f}]  "
          f"TOST p={p_tost:.4f}  →  {'EQUIVALENT (|diff|<0.02)' if equiv else 'NOT equivalent at Δ=0.02'}")
    return {'label': label, 'diff': round(obs, 4), 'ci': [round(lo, 4), round(hi, 4)],
            'p_tost': round(float(p_tost), 4), 'equivalent': bool(equiv), 'margin': MARGIN}

# System accessors (seed -> (s1_preds, s2_preds)); single-pass uses same file twice.
E   = lambda s: (load_preds(find_preds(s, 'joint')),) * 2
E4  = lambda s: (load_preds(f'models/xlmr_2x2/e4_s{s}/test_predictions.jsonl'),) * 2
D   = lambda s: (load_preds(f'models/xlmr_2x2/stage1_s{s}/test_predictions.jsonl'),
                 load_preds(find_preds(s, 'joint')))
DB  = lambda s: (load_preds(f'models/xlmr_2x2/stage1_s{s}/test_predictions.jsonl'),
                 load_preds(find_preds(s, 'bio')))

print("=" * 70)
print(f"Per-example TOST equivalence  (Δ=±{MARGIN}, {N_BOOT} boot, resample-within-language)")
print("=" * 70)
tost_results = [
    tost(build_pairs(E,  E4), "E-XLM-R vs E4-XLM-R  (single-pass QA ≈ BIO?)"),
    tost(build_pairs(D,  DB), "D-XLM-R vs D-BIO-XLM-R (pipeline QA vs BIO)"),
]
print("\nInterpretation vs mBERT (canonical): single-pass E−E4 = +0.0048 TOST-equivalent;")
print("pipeline D−D-BIO = +0.0212 exceeds ±0.02 (QA meaningfully beats BIO in pipeline).")
print("If XLM-R single-pass is EQUIVALENT and pipeline stays within ±0.02, the honest")
print("claim is: direction replicates, but the strong pipeline advantage is mBERT-specific.")

# Persist alongside the 2×2 results
import json
tost_path = Path('models/xlmr_2x2/xlmr_2x2_tost.json')
tost_path.write_text(json.dumps({'margin': MARGIN, 'n_boot': N_BOOT,
                                 'method': 'per-example paired bootstrap, resample within language',
                                 'results': tost_results}, indent=2))
print(f"\n✓ Saved → {tost_path}")